# Reinforcement learning from scratch (numpy only)

Four experiments, each matching a section of `notes.md`:

1. **Value iteration and policy iteration** on a 4x4 gridworld (notes §3):
   the two give the same values and policy, and each value-iteration sweep
   shrinks the distance to the fixed point by a factor of at most gamma.
2. **Monte Carlo vs TD(0)** evaluation of the two-state loop chain (notes §4):
   both converge to V(s1) = 0.818, and TD's estimates vary less from run to run.
3. **Q-learning vs SARSA** on a cliff (notes §4): SARSA learns a path away from
   the cliff, Q-learning the shortest path along its edge.
4. **REINFORCE on a bandit** with and without a baseline (notes §5): the same
   expected gradient, with lower variance when the baseline is subtracted.

Every claim is checked with an `assert`. Run the cells in order: one random
generator with a fixed seed is shared by all experiments, so the printed
numbers are reproducible. Everything runs in a few seconds on CPU.

In [1]:
import numpy as np
rng = np.random.default_rng(0)

## 1. Gridworld: value iteration vs policy iteration

A 4x4 grid with the goal in the bottom-right corner. Entering the goal pays
+1 and ends the episode; every other step pays -0.04. The four actions are
up, down, left and right. An action moves in the intended direction with
probability 0.8 and in each of the two perpendicular directions with
probability 0.1; a move into a wall leaves the agent where it was.

The next cell writes these dynamics as `transitions(s, ai)`, which returns
the list of (probability, next state, reward) triples. Both solvers below
use only this function, which is what "the dynamics are known" means.

In [2]:
N = 4
GOAL = (N - 1, N - 1)
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right
GAMMA = 0.9
STEP_COST = -0.04
STATES = [(r, c) for r in range(N) for c in range(N)]

def move(s, a):
    r, c = s[0] + a[0], s[1] + a[1]
    return s if not (0 <= r < N and 0 <= c < N) else (r, c)

def transitions(s, ai):
    """List of (prob, next_state, reward) for taking action index ai in s."""
    if s == GOAL:
        return [(1.0, s, 0.0)]    # terminal: stays put with reward 0, so V(GOAL) = 0
    a = ACTIONS[ai]
    perp = [(a[1], a[0]), (-a[1], -a[0])]
    out = []
    for p, d in [(0.8, a), (0.1, perp[0]), (0.1, perp[1])]:
        s2 = move(s, d)
        out.append((p, s2, 1.0 if s2 == GOAL else STEP_COST))
    return out

def q_from_v(V):
    """One Bellman backup: Q(s, a) = sum over s' of P(s'|s,a) [R + gamma V(s')]."""
    Q = {}
    for s in STATES:
        for ai in range(4):
            Q[s, ai] = sum(p * (rew + GAMMA * V[s2]) for p, s2, rew in transitions(s, ai))
    return Q

**Value iteration** applies the Bellman optimality update to every state,
starting from V = 0, until no value changes by more than 1e-10. The
contraction argument in notes §3 says the largest gap to the fixed point,
max over s of |V_k(s) - V*(s)|, shrinks by a factor of at most gamma = 0.9
per sweep. The cell records every sweep, uses the final V as V*, and prints
the largest observed ratio of successive gaps, which should not exceed 0.9.

In [3]:
def value_iteration(tol=1e-10):
    V = {s: 0.0 for s in STATES}
    history = []
    while True:
        Q = q_from_v(V)
        V_new = {s: max(Q[s, ai] for ai in range(4)) for s in STATES}
        delta = max(abs(V_new[s] - V[s]) for s in STATES)
        V = V_new
        history.append(dict(V))
        if delta < tol:
            return V, history

V_vi, hist = value_iteration()
print("value iteration sweeps:", len(hist))

dist = [max(abs(Vk[s] - V_vi[s]) for s in STATES) for Vk in hist[:-1]]
ratios = [dist[i + 1] / dist[i] for i in range(len(dist) - 1) if dist[i] > 1e-8]
print("largest ratio of successive distances to V*:", round(max(ratios), 3),
      "(the theorem guarantees at most gamma = 0.9)")
assert max(ratios) <= GAMMA + 1e-9

value iteration sweeps: 37
largest ratio of successive distances to V*: 0.885 (the theorem guarantees at most gamma = 0.9)


The observed ratios never exceed 0.9; the theorem gives an upper bound, and
on this grid the ratio is below it.

**Policy iteration** starts from the policy "always up" and alternates two
steps. Evaluation solves the linear system (I - gamma P_pi) V = R_pi exactly,
one equation per state. Improvement switches every state to the greedy
action under that V. The loop stops when the policy no longer changes. The
result should match value iteration's values to floating-point precision,
and the printed policy should point every arrow toward the goal.

In [4]:
def policy_iteration():
    pi = {s: 0 for s in STATES}
    idx = {s: i for i, s in enumerate(STATES)}
    while True:
        # evaluate: solve the linear system (I - gamma P_pi) V = R_pi
        A = np.eye(len(STATES)); b = np.zeros(len(STATES))
        for s in STATES:
            for p, s2, rew in transitions(s, pi[s]):
                A[idx[s], idx[s2]] -= GAMMA * p
                b[idx[s]] += p * rew
        v = np.linalg.solve(A, b)
        V = {s: v[idx[s]] for s in STATES}
        # improve: greedy action under V
        Q = q_from_v(V)
        pi_new = {s: max(range(4), key=lambda ai: Q[s, ai]) for s in STATES}
        if pi_new == pi:
            return V, pi
        pi = pi_new

V_pi, pi_star = policy_iteration()
gap = max(abs(V_vi[s] - V_pi[s]) for s in STATES)
print("max |V_vi - V_pi| =", gap)
assert gap < 1e-8

arrows = {0: "^", 1: "v", 2: "<", 3: ">"}
for r in range(N):
    print(" ".join("G" if (r, c) == GOAL else arrows[pi_star[r, c]] for c in range(N)))

max |V_vi - V_pi| = 7.676442814741335e-11
v > v v
v v v v
> > v v
> > > G


Both methods give the same value function (the difference is at
floating-point level) and the same policy: every arrow is down or right,
toward the goal.

## 2. Monte Carlo vs TD(0) on the loop chain

This is the chain from notes §2. From s1 the agent stays in s1 with
probability 0.5 or moves to s2 with probability 0.5, with reward 0 either
way. From s2 it moves to the terminal state with reward 1. With gamma = 0.9
the exact value is V(s1) = 0.45/0.55 = 0.818.

The next cell defines the episode sampler and the two estimators.
`mc_estimate` moves V(s) toward the full observed return G_t. `td_estimate`
moves V(s) toward the one-step target r + gamma V(s'). Both use step size
alpha = 0.05.

In [5]:
GAMMA_CHAIN = 0.9

def episode():
    """Returns the list of (state, reward) pairs, states 0 = s1, 1 = s2."""
    traj = []
    s = 0
    while True:
        if s == 0:
            traj.append((0, 0.0))
            s = 0 if rng.random() < 0.5 else 1
        else:
            traj.append((1, 1.0))
            return traj

def mc_estimate(n_episodes, alpha=0.05):
    V = np.zeros(2)
    for _ in range(n_episodes):
        traj = episode()
        Gt = 0.0
        for s, r in reversed(traj):          # accumulate the return backward
            Gt = r + GAMMA_CHAIN * Gt
            V[s] += alpha * (Gt - V[s])
    return V

def td_estimate(n_episodes, alpha=0.05):
    V = np.zeros(3)   # index 2 = terminal, value 0
    for _ in range(n_episodes):
        traj = episode()
        for i, (s, r) in enumerate(traj):
            s2 = traj[i + 1][0] if i + 1 < len(traj) else 2
            V[s] += alpha * (r + GAMMA_CHAIN * V[s2] - V[s])
    return V[:2]

Now run each estimator 20 times, 2000 episodes per run, and compare the
final estimates of V(s1). Both means should be close to 0.818. The standard
deviation across the 20 runs measures how noisy each estimator is at the
same step size; TD's should be smaller, because its target contains one
random reward and one estimate instead of a whole episode's return.

In [6]:
runs_mc = np.array([mc_estimate(2000)[0] for _ in range(20)])
runs_td = np.array([td_estimate(2000)[0] for _ in range(20)])
exact = 0.45 / 0.55
print(f"exact V(s1) = {exact:.4f}")
print(f"MC : mean {runs_mc.mean():.4f}, std across runs {runs_mc.std():.4f}")
print(f"TD : mean {runs_td.mean():.4f}, std across runs {runs_td.std():.4f}")
assert abs(runs_mc.mean() - exact) < 0.03 and abs(runs_td.mean() - exact) < 0.03
assert runs_td.std() < runs_mc.std()

exact V(s1) = 0.8182
MC : mean 0.8148, std across runs 0.0291
TD : mean 0.8239, std across runs 0.0157


Both are centered near 0.818, and TD's spread across runs is about half of
Monte Carlo's.

## 3. Q-learning vs SARSA on the cliff

A grid of 4 rows and 8 columns. The start is the bottom-left cell and the
goal the bottom-right cell; the cells between them on the bottom row are the
cliff. Stepping onto the cliff pays -100 and returns the agent to the start;
every other step pays -1. Both algorithms train for 600 episodes with
epsilon-greedy exploration, epsilon = 0.1 (a random action 10% of the time,
the highest-Q action otherwise), step size 0.5 and gamma = 1.

The next cell defines the environment and one training loop for both
algorithms. The only difference is the target: Q-learning uses the largest
Q at the next state, SARSA uses the Q of the action it actually takes next.

In [7]:
ROWS, COLS = 4, 8
START, GOALC = (ROWS - 1, 0), (ROWS - 1, COLS - 1)
CLIFF = {(ROWS - 1, c) for c in range(1, COLS - 1)}

def cliff_step(s, ai):
    a = ACTIONS[ai]
    r, c = min(max(s[0] + a[0], 0), ROWS - 1), min(max(s[1] + a[1], 0), COLS - 1)
    s2 = (r, c)
    if s2 in CLIFF:
        return START, -100.0, False
    return s2, -1.0, s2 == GOALC

def eps_greedy(Q, s, eps):
    if rng.random() < eps:
        return rng.integers(4)
    return int(np.argmax(Q[s]))

def train(kind, episodes=600, alpha=0.5, eps=0.1, gamma=1.0):
    Q = np.zeros((ROWS, COLS, 4))
    returns = []
    for _ in range(episodes):
        s = START; a = eps_greedy(Q, s, eps); total = 0.0
        while True:
            s2, r, done = cliff_step(s, a)
            total += r
            a2 = eps_greedy(Q, s2, eps)
            if done:
                target = r
            elif kind == "qlearning":
                target = r + gamma * Q[s2].max()     # best next action
            else:
                target = r + gamma * Q[s2][a2]       # action actually taken next
            Q[s][a] += alpha * (target - Q[s][a])
            s, a = s2, a2
            if done:
                break
        returns.append(total)
    return Q, np.array(returns)

After training, follow each learned Q greedily (no random actions) from the
start and draw the path: `*` marks visited cells, `C` the cliff. Expected
result: Q-learning's path runs along the row directly above the cliff, the
shortest route. SARSA's path climbs away from the cliff before crossing,
and SARSA earns the higher average return over the last 100 training
episodes, because those episodes still include random actions.

In [8]:
def greedy_path(Q):
    s = START; path = [s]
    for _ in range(50):
        s, _, done = cliff_step(s, int(np.argmax(Q[s])))
        path.append(s)
        if done:
            break
    return path

def show(Q, title):
    path = set(greedy_path(Q))
    print(title)
    for r in range(ROWS):
        row = ""
        for c in range(COLS):
            cell = (r, c)
            row += ("C" if cell in CLIFF else "S" if cell == START else
                    "G" if cell == GOALC else "*" if cell in path else ".")
        print("  " + row)

Q_ql, ret_ql = train("qlearning")
Q_sa, ret_sa = train("sarsa")
show(Q_ql, "Q-learning greedy path (the row next to the cliff):")
show(Q_sa, "SARSA greedy path (farther from the cliff):")
print("mean return over the last 100 training episodes, with exploration on:")
print(f"  Q-learning {ret_ql[-100:].mean():7.1f}   SARSA {ret_sa[-100:].mean():7.1f}")

path_ql, path_sa = greedy_path(Q_ql), greedy_path(Q_sa)
assert {(ROWS - 2, c) for c in range(COLS)} <= set(path_ql)   # edge row, end to end
assert min(r for r, _ in path_sa) < ROWS - 2                  # SARSA moves higher up
assert len(path_ql) < len(path_sa)                            # Q-learning's path is shorter
assert ret_sa[-100:].mean() > ret_ql[-100:].mean()

Q-learning greedy path (the row next to the cliff):
  ........
  ........
  ********
  SCCCCCCG
SARSA greedy path (farther from the cliff):
  .******.
  **....*.
  *.....**
  SCCCCCCG
mean return over the last 100 training episodes, with exploration on:
  Q-learning   -35.9   SARSA   -19.2


Q-learning finds the shortest path, which is optimal for an agent that never
takes random actions. SARSA learns the value of the policy it actually
follows, random actions included, so it picks a path where a random step
cannot reach the cliff, and it earns a better return while exploring.

## 4. REINFORCE on a bandit: the baseline lowers variance

Five arms; arm k pays 1 with probability [0.1, 0.3, 0.5, 0.7, 0.9][k] and 0
otherwise. The policy is a softmax over five logits, all starting at 0, so
every arm has probability 0.2. For a softmax, the gradient of log pi(a) is
the one-hot vector of a minus pi (notes §5).

The next cell draws 20000 single-sample gradient estimates (r - b)(e_a - pi)
at this fixed policy, once with b = 0 and once with b equal to the expected
reward under the policy (0.5). Expected result: the two mean gradients agree,
as the baseline lemma says, and the baseline version has a smaller total
variance.

In [9]:
P_ARMS = np.array([0.1, 0.3, 0.5, 0.7, 0.9])

def softmax(z):
    z = z - z.max(); e = np.exp(z); return e / e.sum()

def grad_samples(theta, n, baseline):
    pi = softmax(theta)
    grads = np.zeros((n, 5))
    for i in range(n):
        a = rng.choice(5, p=pi)
        r = float(rng.random() < P_ARMS[a])
        onehot = np.zeros(5); onehot[a] = 1
        grads[i] = (r - baseline) * (onehot - pi)
    return grads

theta = np.zeros(5)
pi0 = softmax(theta)
true_b = (pi0 * P_ARMS).sum()          # expected reward under the current policy
g_plain = grad_samples(theta, 20000, 0.0)
g_base = grad_samples(theta, 20000, true_b)
print("mean gradient, no baseline :", np.round(g_plain.mean(0), 3))
print("mean gradient, baseline    :", np.round(g_base.mean(0), 3))
print(f"total variance, no baseline: {g_plain.var(0).sum():.4f}")
print(f"total variance, baseline   : {g_base.var(0).sum():.4f}")
assert np.allclose(g_plain.mean(0), g_base.mean(0), atol=0.01)
assert g_base.var(0).sum() < g_plain.var(0).sum()

mean gradient, no baseline : [-0.079 -0.04  -0.002  0.039  0.082]
mean gradient, baseline    : [-0.081 -0.04  -0.001  0.042  0.08 ]
total variance, no baseline: 0.3800
total variance, baseline   : 0.1837


The two mean gradients agree: positive for the arms that pay more than
average, negative for the others. The baseline roughly halves the total
variance.

Finally, run REINFORCE with a baseline for 3000 steps, learning rate 0.1.
The baseline here is the running mean of all rewards seen so far, since the
true expected reward is unknown during learning. Expected result: the
policy puts nearly all of its probability on the best arm (arm 5, which
pays with probability 0.9).

In [10]:
theta = np.zeros(5); avg_r = 0.0; lr = 0.1
for t in range(1, 3001):
    pi = softmax(theta)
    a = rng.choice(5, p=pi)
    r = float(rng.random() < P_ARMS[a])
    avg_r += (r - avg_r) / t                # running mean reward as the baseline
    onehot = np.zeros(5); onehot[a] = 1
    theta += lr * (r - avg_r) * (onehot - pi)
print("final policy:", np.round(softmax(theta), 3))
print("probability on the best arm:", round(float(softmax(theta)[4]), 3))
assert softmax(theta)[4] > 0.9

final policy: [0.001 0.001 0.002 0.003 0.993]
probability on the best arm: 0.993
